# Phase 1 Deliverable: Epidemiology Data Pipeline Tool

**Industry Practicum – PharmaACE**  
*Making epidemiology trustworthy enough to act on: source-backed, validated evidence.*

## Objective

Phase 1 delivers a **multi-source evidence finder** and **data pipeline** that:

1. **Aggregates epidemiology evidence** from many public sources (registries, guidelines, journals, trials, dashboards) for a given **indication** and **country**.
2. **Extracts datapoints** (incidence, prevalence, rates, counts) where possible via configurable deep-dive extraction; otherwise provides **source links** for manual lookup.
3. **Produces tool-ready outputs**: evidence table, reference links, KPI table, and optional dashboard layer for downstream use (Excel, Power BI, Tableau, InsightACE).

This notebook demonstrates the **pipeline architecture**, **configuration**, and **sample outputs** for the Phase 1 deliverable.

## 1. Pipeline Architecture

```
  [Config] sources_to_explore.yaml, required_metrics.yaml
       |
       v
  Evidence Finder  -->  Evidence CSV + Source log
  (link deep-dive + API connectors)
       |
       v
  Data Builder     -->  Tool-ready CSV, InsightACE export
       |
       v
  KPI Table        -->  KPI CSV, conflicts
       |
       v
  Dashboard export -->  CSV + SQLite for Tableau / Power BI
```

- **Evidence Finder**: Loads all sources from `config/sources_to_explore.yaml`; for each link source, optionally deep-dives to extract numbers (with 45-min cap and priority ordering). API sources (ClinicalTrials, PubMed) are queried live.
- **Data Builder**: Converts evidence to tool-ready schema with scenario options.
- **KPI Table**: Required metrics (e.g. incidence, prevalence) aligned with NCCN/InsightACE; tracks gaps and conflicts.
- **Dashboard**: Exports a single folder (CSV + SQLite) for BI tools and forecasting/insights.

In [ ]:
# Setup: add project root to path and imports
import sys
from pathlib import Path

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import yaml

try:
    from IPython.display import display
except ImportError:
    display = print

CONFIG_DIR = ROOT / "config"
OUTPUT_DIR = ROOT / "output"
print(f"Project root: {ROOT}")
print(f"Config dir:   {CONFIG_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")

## 2. Configuration Overview

**Sources** (`sources_to_explore.yaml`): Link-type sources (URL templates with `{indication}`, `{country}`) and API-type sources (ClinicalTrials, PubMed). Priority ordering ensures high-value sources (GLOBOCAN, WHO, NCCN, SEER, CDC, etc.) are deep-dived first within the time cap.

**Required metrics** (`required_metrics_*.yaml`): KPI definitions per indication (e.g. incidence, prevalence) for NCCN/InsightACE alignment.

In [ ]:
# Load sources to explore
with open(CONFIG_DIR / "sources_to_explore.yaml", "r") as f:
    sources_config = yaml.safe_load(f)

sources = sources_config.get("sources") or []
link_sources = [s for s in sources if s.get("type") == "link"]
api_sources = [s for s in sources if s.get("type") == "api"]

print(f"Total sources: {len(sources)}")
print(f"  Link sources: {len(link_sources)}")
print(f"  API sources:  {len(api_sources)}")
print()
print("Sample link sources (name, tier):")
for s in link_sources[:10]:
    print(f"  - {s.get('name', '?')} ({s.get('tier', '?')})")

In [ ]:
# Load required metrics for CLL
metrics_path = CONFIG_DIR / "required_metrics_cll.yaml"
with open(metrics_path, "r") as f:
    metrics_config = yaml.safe_load(f)

metrics = metrics_config.get("metrics") or []
print(f"Required metrics for {metrics_config.get('indication', '?')}:")
for m in metrics:
    print(f"  - {m.get('id')}: {m.get('label')} (required={m.get('required', False)}")

## 3. Running the Pipeline (Optional)

The full pipeline can be run via the **web app** (`python app_web.py` → http://127.0.0.1:5000) or programmatically below. For this deliverable we **load existing output** if present so the notebook runs quickly without a long scrape.

In [ ]:
# Option A: Load existing evidence output (no run)
# Single evidence file is evidence_by_metric_*.csv (sorted by metric); fall back to evidence_*.csv for older runs
candidates = list(OUTPUT_DIR.glob("evidence_by_metric_*.csv")) or list(OUTPUT_DIR.glob("evidence_*.csv"))
preferred = [p for p in candidates if "CLL" in p.name and "US" in p.name]
EVIDENCE_FILE = preferred[0] if preferred else (candidates[0] if candidates else None)
evidence_df = None
INDICATION_SUFFIX = ""

def add_year_column_if_missing(df):
    """Add a 'year' column parsed from 'year_or_range' so which year's data is explicit."""
    if df is None or df.empty or "year" in df.columns:
        return df
    if "year_or_range" not in df.columns:
        df = df.copy()
        df["year"] = pd.NA
        return df
    def parse_year(val):
        if pd.isna(val) or val is None: return None
        s = str(val).strip()
        if not s: return None
        for part in s.replace("-", " ").split():
            if len(part) >= 4 and part[:4].isdigit():
                return int(part[:4])
        return None
    out = df.copy()
    out["year"] = df["year_or_range"].apply(parse_year)
    return out

if EVIDENCE_FILE and EVIDENCE_FILE.exists():
    evidence_df = pd.read_csv(EVIDENCE_FILE)
    evidence_df = add_year_column_if_missing(evidence_df)
    INDICATION_SUFFIX = EVIDENCE_FILE.stem.replace("evidence_by_metric_", "").replace("evidence_", "")
    print(f"Loaded evidence: {EVIDENCE_FILE.name}")
    print(f"Rows: {len(evidence_df)}")
else:
    print("No existing evidence file found. Run the pipeline via the web app or uncomment Option B below.")

In [ ]:
# Option B: Run pipeline (takes several minutes; 45-min cap)
# Uncomment to execute.
# from src.pipeline.runner import run_pipeline
# result = run_pipeline(
#     indication="CLL (Chronic Lymphocytic Leukemia)",
#     country="US",
#     config_dir=CONFIG_DIR,
#     output_dir=OUTPUT_DIR,
#     export_dashboard=True,
#     use_pubmed=True,
#     max_run_seconds=45*60,
# )
# print(result.get("message", ""))
# evidence_df = pd.read_csv(OUTPUT_DIR / f"evidence_by_metric_{INDICATION_SUFFIX}.csv") if result.get("success") else None
# if evidence_df is not None: evidence_df = add_year_column_if_missing(evidence_df)

## 4. Evidence Table Summary

The evidence table has one row per source. The **value** column either contains **extracted datapoints** (numbers, rates, percentages) or **"See link for incidence, prevalence, and KPIs"** with the URL in **source_url** for manual lookup. **notes** indicate whether extraction was attempted or skipped (e.g. time limit).

In [ ]:
if evidence_df is not None:
    # Summary: extracted vs "See link"
    has_extracted = evidence_df["value"].astype(str).str.contains("See link", na=True) == False
    print("Evidence summary:")
    print(f"  Sources with extracted values: {has_extracted.sum()}")
    print(f"  Sources with stub (See link):   {(~has_extracted).sum()}")
    # Show year column separately so which year's data is clear
    cols = ["source_citation", "source_tier", "metric", "year_or_range", "value"]
    if "year" in evidence_df.columns:
        cols.insert(cols.index("year_or_range") + 1, "year")
    display(evidence_df[[c for c in cols if c in evidence_df.columns]].head(12))
else:
    print("No evidence data to display.")

## Part 1. Confidence scoring & rubric

Each evidence row gets a **computed confidence score** (0–100) and label (High/Medium/Low) from: source tier, extraction success, completeness (definition, **year**, geography, population), and recency. The **year** column (parsed from `year_or_range`) shows which year's data each row refers to.

In [ ]:
# Part 1: Confidence-scored evidence and rubric
if evidence_df is not None:
    conf_cols = [c for c in ["metric", "year_or_range", "year", "value", "source_citation", "source_tier"] if c in evidence_df.columns]
    if "computed_confidence_score" in evidence_df.columns:
        conf_cols.append("computed_confidence_score")
    if "computed_confidence" in evidence_df.columns:
        conf_cols.append("computed_confidence")
    display(evidence_df[[c for c in conf_cols]][:12])
rubric_path = OUTPUT_DIR / "confidence_rubric.md"
if rubric_path.exists():
    print("\nConfidence rubric (excerpt):")
    print(rubric_path.read_text()[:1200])
else:
    print("Run the pipeline to generate confidence_rubric.md.")

## Part 2. KPI scorecard

Scorecard per required metric: coverage (n sources, best value, source), agreement, and validation status (Ready / Needs review / No source). Year is shown where relevant for time-bound metrics.

In [ ]:
# Part 2: KPI scorecard
SCORECARD_FILE = OUTPUT_DIR / f"kpi_scorecard_{INDICATION_SUFFIX}.csv" if INDICATION_SUFFIX else None
if SCORECARD_FILE and SCORECARD_FILE.exists():
    scorecard_df = pd.read_csv(SCORECARD_FILE)
    print(f"KPI scorecard: {SCORECARD_FILE.name}")
    display(scorecard_df)
else:
    print("KPI scorecard not found. Run the pipeline to generate kpi_scorecard_*.csv.")

## Part 3. White-space map

Coverage matrix: metrics × geography/year — where we have strong evidence (Has value), stub only, or missing. Extracts and shows gaps clearly.

In [ ]:
# Part 3: White-space summary
WS_MD = OUTPUT_DIR / f"white_space_{INDICATION_SUFFIX}_summary.md" if INDICATION_SUFFIX else None
if WS_MD and WS_MD.exists():
    print("White-space summary:")
    print(WS_MD.read_text())
else:
    print("White-space summary not found. Run the pipeline to generate it.")

## Part 4. Source agreement & reconciliation

For metrics with multiple source values, the reconciliation table shows: **year_or_range** (which year's data), value by source, range, and recommended value. Year is kept separate so you can see which year each row refers to.

In [ ]:
# Part 4: Reconciliation (with year column)
RECON_FILE = OUTPUT_DIR / f"reconciliation_{INDICATION_SUFFIX}.csv" if INDICATION_SUFFIX else None
if RECON_FILE and RECON_FILE.exists():
    recon_df = pd.read_csv(RECON_FILE)
    print(f"Reconciliation: {RECON_FILE.name}")
    # year_or_range shows which year's data
    display(recon_df)
else:
    print("Reconciliation file not found. Run the pipeline; it is generated when multiple sources report the same metric/year.")

### Extracted data with year column

Tool-ready table: one row per evidence datapoint with **year** kept separate so you can see which year each value refers to.

In [ ]:
# Tool-ready (extracted data) — year column shows which year's data
TOOL_READY_FILE = OUTPUT_DIR / f"tool_ready_{INDICATION_SUFFIX}.csv" if INDICATION_SUFFIX else None
if TOOL_READY_FILE and TOOL_READY_FILE.exists():
    tool_df = pd.read_csv(TOOL_READY_FILE)
    print("Tool-ready table (year column = which year's data):")
    display(tool_df.head(15))
else:
    print("Tool-ready file not found. Run the pipeline to generate it.")

## 5. KPI Table (if available)

The KPI table summarizes required metrics with evidence coverage and conflicts.

In [ ]:
KPI_FILE = OUTPUT_DIR / f"kpi_scorecard_{INDICATION_SUFFIX}.csv" if INDICATION_SUFFIX else None
if KPI_FILE and KPI_FILE.exists() and evidence_df is not None:
    kpi_df = pd.read_csv(KPI_FILE)
    print(f"KPI scorecard: {KPI_FILE.name}")
    display(kpi_df.head(10))
else:
    print("KPI scorecard not found or evidence not loaded.")

## 6. Outputs Delivered

| Output | Description |
|--------|-------------|
| **evidence_by_metric_*.csv** | Evidence sorted by metric; one row per source; value = extracted or "See link"; source_url for manual lookup |
| **reference_links_*.csv** | All link-type sources with URL and metric for reference |
| **tool_ready_*.csv** | Tool-ready schema for downstream systems |
| **kpi_scorecard_*.csv** | Per-metric coverage, best value, agreement, validation status; **kpi_conflicts_*.csv** when conflicts exist |
| **dashboard/** | CSV + SQLite for Tableau / Power BI, forecast, insights |

---

## Phase 1 Summary

- **Evidence Finder** aggregates 70+ configured sources (registries, guidelines, journals, trials, dashboards) and performs bounded deep-dive extraction with priority and time cap.
- **Config-driven** sources and metrics; proxy and scraper settings supported for robustness.
- **Tool-ready and KPI outputs** align with NCCN/InsightACE and support dashboard connectivity.

**Next (Phase 2+):** Deeper integration with BI tools, additional connectors, and client-specific refinements.